# Prédiction du churn client — Modélisation

Suite de `01_exploration.ipynb`. On compare 3 modèles (régression logistique,
Random Forest, XGBoost) pour repérer les clients qui risquent de partir.

## 1. Imports et chargement

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

On reprend le nettoyage de l'exploration, puis on sépare les variables (X) de la cible (y).

In [23]:
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]
print(X.shape, y.shape)

(7043, 19) (7043,)


## 2. Découpage train / test

80 % pour l'entraînement, 20 % pour le test. `stratify=y` garde le même taux de churn (26,5 %) dans les deux parties.

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [25]:
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(5634, 19) (1409, 19)
0.2653532126375577 0.2654364797728886


## 3. Préparation des données

- Colonnes texte → one-hot encoding (une colonne 0/1 par modalité)
- Colonnes numériques → mise à l'échelle (`StandardScaler`)

Le tout est regroupé dans un `ColumnTransformer`, intégré ensuite dans chaque Pipeline.

In [26]:
cat_cols = X_train.select_dtypes(include="object").columns
num_cols = X_train.select_dtypes(exclude="object").columns
print(len(cat_cols), "colonnes texte :", list(cat_cols))
print(len(num_cols), "colonnes numériques :", list(num_cols))

15 colonnes texte : ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
4 colonnes numériques : ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


C:\Users\yacin\AppData\Local\Temp\ipykernel_9448\2161354818.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns


In [27]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

## 4. Modèles

### Régression logistique

In [28]:
model_lr = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000)),
])

model_lr.fit(X_train, y_train)

print(model_lr.score(X_test, y_test))

0.8055358410220014


In [29]:
y_pred = model_lr.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



Accuracy de 0,81, mais trompeuse : prédire toujours « reste » donnerait déjà 0,735.

Le recall des churners n'est que de 0,56, donc presque un départ sur deux est raté.

→ On ajoute `class_weight="balanced"` pour donner plus de poids aux churners.

In [30]:
model_lr_bal = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

model_lr_bal.fit(X_train, y_train)
y_pred_bal = model_lr_bal.predict(X_test)

print(classification_report(y_test, y_pred_bal))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



On choisit la version "balanced" (recall à 0,78) : rater un client qui part coûte plus cher qu'envoyer une offre pour rien. En contrepartie, la precision baisse à 0,50.

### Random Forest

In [31]:
model_rf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
    )),
])

model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.86      0.82      0.84      1035
           1       0.56      0.63      0.59       374

    accuracy                           0.77      1409
   macro avg       0.71      0.73      0.72      1409
weighted avg       0.78      0.77      0.77      1409



In [32]:
print("Train :", model_rf.score(X_train, y_train))
print("Test  :", model_rf.score(X_test, y_test))

Train : 0.9941427050053249
Test  : 0.7700496806245565


In [33]:
model_rf2 = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
    )),
])

model_rf2.fit(X_train, y_train)

print("Train :", model_rf2.score(X_train, y_train))
print("Test  :", model_rf2.score(X_test, y_test))
print(classification_report(y_test, model_rf2.predict(X_test)))

Train : 0.7783102591409301
Test  : 0.7544357700496807
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1035
           1       0.52      0.80      0.63       374

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.75      0.77      1409



**Correction du surapprentissage**

Le Random Forest par défaut a un recall de 0,63, inférieur à la régression logistique (0,78).

-> Surapprentissage. Accuracy de 0,99 sur le train contre 0,77 sur le test.

**Correction** : `min_samples_leaf=20`. Chaque "feuille" doit contenir au moins 20 clients, ce qui force des règles plus générales.

**Résultat** : écart train/test réduit (0,77 / 0,75), recall à 0,80, precision à 0,52.
Légèrement meilleur que la régression logistique, mais moins simple à expliquer.

### XGBoost

`scale_pos_weight` joue le même rôle que `class_weight="balanced"` :
le ratio fidèles / churners (≈ 2,77) donne plus de poids aux churners.

In [34]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(ratio)

2.768561872909699


In [35]:
model_xgb = Pipeline([
    ("prep", preprocessor),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=ratio,
        random_state=42,
    )),
])

model_xgb.fit(X_train, y_train)

print("Train :", model_xgb.score(X_train, y_train))
print("Test  :", model_xgb.score(X_test, y_test))
print(classification_report(y_test, model_xgb.predict(X_test)))

Train : 0.8081292154774583
Test  : 0.751596877217885
              precision    recall  f1-score   support

           0       0.90      0.74      0.82      1035
           1       0.52      0.77      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409



Precision 0,52, recall 0,77. Écart train/test d'environ 6 points (0,81 / 0,75) :
léger surapprentissage, acceptable.

## 5. Comparaison des modèles

In [36]:
models = {
    "Régression logistique": model_lr_bal,
    "Random Forest": model_rf2,
    "XGBoost": model_xgb,
}

In [37]:
rows = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    rows.append({
        "Modèle": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

In [38]:
results = pd.DataFrame(rows).set_index("Modèle").round(3)
results

,Precision,Recall,F1,ROC-AUC
Modèle,,,,
Régression logistique,0.504,0.783,0.614,0.842
Random Forest,0.525,0.797,0.633,0.844
XGBoost,0.522,0.770,0.622,0.840


Les 3 modèles atteignent un ROC-AUC d'environ 0,84 : les performances sont équivalentes.

→ On choisit la **régression logistique** : aussi performante, et plus facile à expliquer au métier.

## 6. Interprétation du modèle

Coefficients de la régression logistique : positif = augmente le risque de départ,
négatif = le diminue.

In [39]:
feature_names = model_lr_bal.named_steps["prep"].get_feature_names_out()
coefs = model_lr_bal.named_steps["clf"].coef_[0]
coef_df = pd.DataFrame({"Variable": feature_names, "Coefficient": coefs})
coef_df = coef_df.sort_values("Coefficient")

print("Augmentent le risque :")
print(coef_df.tail(5))
print("\nDiminuent le risque :")
print(coef_df.head(5))

Augmentent le risque :
                            Variable  Coefficient
32              cat__StreamingTV_Yes     0.262166
35          cat__StreamingMovies_Yes     0.275392
3                  num__TotalCharges     0.489698
36      cat__Contract_Month-to-month     0.657713
16  cat__InternetService_Fiber optic     0.707981

Diminuent le risque :
                                     Variable  Coefficient
1                                 num__tenure    -1.154434
38                     cat__Contract_Two year    -0.775518
2                         num__MonthlyCharges    -0.676114
15                   cat__InternetService_DSL    -0.622465
25  cat__DeviceProtection_No internet service    -0.275403


Les valeurs obtenues confirment l'exploration :
- `Fiber optic` (+0,71) → la fibre augmente le risque.
- `Month-to-month` (+0,66) → le contrat mensuel augmente le risque.
- `tenure` (−1,15) → la variable la plus forte de toutes. Plus le client est ancien, moins il part.
- `Two year` (−0,78) → le contrat 2 ans protège.

`MonthlyCharges` est négatif, ce qui contredit le boxplot. Cause : `TotalCharges` ≈ `tenure` × `MonthlyCharges`.
Ces variables sont trop liées, donc leurs coefficients pris séparément ne sont pas fiables.